# 01 - Exploratory Data Analysis (Bronze layer)

**Phase 3 goal:** understand the raw data and record every data issue plus the decision we make for it, *before* building dbt models.

**How to use this notebook**
1. Run all cells top to bottom (the warehouse is opened read-only).
2. For each section, note anything surprising in `docs/data_findings.md`.
3. Do not clean data here. Decisions are recorded, then implemented in dbt staging (Phase 4).

All raw columns are text, so casts use `TRY_CAST` (bad values become NULL instead of failing).

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px

# Make the project importable when running from analytics/notebooks
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from ingestion.config import TABLES, get_warehouse_path  # noqa: E402

DB_PATH = get_warehouse_path()
con = duckdb.connect(str(DB_PATH), read_only=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


def q(sql: str, params: list | None = None) -> pd.DataFrame:
    """Run SQL and return a DataFrame."""
    return con.execute(sql, params or []).df()


print("Warehouse:", DB_PATH)

## 1. Row counts and load metadata
Confirms all 9 tables loaded and when.

In [ ]:
counts = pd.DataFrame(
    [
        (t.name, con.execute(f"SELECT COUNT(*) FROM raw.{t.name}").fetchone()[0], t.filename)
        for t in TABLES
    ],
    columns=["table", "rows", "source_file"],
)
display(counts)
display(q("SELECT MIN(_loaded_at) AS first_loaded, MAX(_loaded_at) AS last_loaded FROM raw.orders"))

## 2. Null / blank rates
Only columns with at least one NULL or empty value are shown.
**Look for:** columns that are mostly empty (e.g. review comments) and date columns that are empty for a reason (e.g. undelivered orders have no delivery date).

In [ ]:
rows = []
for t in TABLES:
    cols = [
        r[0]
        for r in con.execute(
            "SELECT column_name FROM information_schema.columns "
            "WHERE table_schema = 'raw' AND table_name = ? ORDER BY ordinal_position",
            [t.name],
        ).fetchall()
        if not r[0].startswith("_")
    ]
    exprs = ", ".join(
        f'SUM(CASE WHEN "{c}" IS NULL OR TRIM("{c}") = \'\' THEN 1 ELSE 0 END)' for c in cols
    )
    total, *nulls = con.execute(f"SELECT COUNT(*), {exprs} FROM raw.{t.name}").fetchone()
    for c, n in zip(cols, nulls):
        if n:
            rows.append((t.name, c, int(n), int(total), 100 * int(n) / int(total)))

null_report = pd.DataFrame(rows, columns=["table", "column", "null_rows", "total_rows", "null_pct"])
display(null_report.sort_values("null_pct", ascending=False).reset_index(drop=True))

## 3. Key uniqueness
Checks whether each table's expected grain holds. `duplicate_rows = rows - distinct keys`.
**Look for:** `customers.customer_id` should be unique, while `customer_unique_id` should repeat (one real person, many `customer_id`s). Orders, products and sellers should be unique.

In [ ]:
def key_check(table: str, cols: list[str]) -> dict:
    key = ", ".join(cols)
    total = con.execute(f"SELECT COUNT(*) FROM raw.{table}").fetchone()[0]
    distinct = con.execute(
        f"SELECT COUNT(*) FROM (SELECT DISTINCT {key} FROM raw.{table})"
    ).fetchone()[0]
    return {"table": table, "key": key, "rows": total, "distinct_keys": distinct,
            "duplicate_rows": total - distinct}


checks = [
    ("customers", ["customer_id"]),
    ("customers", ["customer_unique_id"]),
    ("orders", ["order_id"]),
    ("order_items", ["order_id", "order_item_id"]),
    ("order_payments", ["order_id", "payment_sequential"]),
    ("order_reviews", ["review_id"]),
    ("order_reviews", ["order_id"]),
    ("products", ["product_id"]),
    ("sellers", ["seller_id"]),
    ("category_translation", ["product_category_name"]),
]
display(pd.DataFrame([key_check(t, c) for t, c in checks]))

## 4. Referential integrity (orphan checks)
Child rows whose parent key does not exist. Any non-zero count needs a decision (drop, keep, or flag).

In [ ]:
def orphans(child: str, ccol: str, parent: str, pcol: str) -> dict:
    n = con.execute(
        f"""
        SELECT COUNT(*) FROM raw.{child} c
        LEFT JOIN raw.{parent} p ON c.{ccol} = p.{pcol}
        WHERE p.{pcol} IS NULL AND c.{ccol} IS NOT NULL AND TRIM(c.{ccol}) <> ''
        """
    ).fetchone()[0]
    return {"child": f"{child}.{ccol}", "parent": f"{parent}.{pcol}", "orphan_rows": n}


pairs = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("order_payments", "order_id", "orders", "order_id"),
    ("order_reviews", "order_id", "orders", "order_id"),
    ("products", "product_category_name", "category_translation", "product_category_name"),
]
display(pd.DataFrame([orphans(*p) for p in pairs]))

## 5. Time coverage and usable window
**Look for:** months with very few orders at the start or end of the data. Sparse months distort trends and forecasts, so we decide a window to analyse.

In [ ]:
monthly = q("""
    SELECT strftime(date_trunc('month', TRY_CAST(order_purchase_timestamp AS TIMESTAMP)), '%Y-%m') AS month,
           COUNT(*) AS orders,
           COUNT(*) FILTER (WHERE order_status = 'delivered') AS delivered_orders
    FROM raw.orders
    GROUP BY 1 ORDER BY 1
""")
display(monthly)
display(q("""
    SELECT MIN(TRY_CAST(order_purchase_timestamp AS TIMESTAMP)) AS first_order,
           MAX(TRY_CAST(order_purchase_timestamp AS TIMESTAMP)) AS last_order,
           COUNT(*) FILTER (WHERE TRY_CAST(order_purchase_timestamp AS TIMESTAMP) IS NULL) AS unparseable
    FROM raw.orders
"""))
px.bar(monthly, x="month", y="orders", title="Orders per month").show()

## 6. Order status mix
**Look for:** the share of `delivered` vs `canceled`, `unavailable`, etc. This decides which orders count toward revenue.

In [ ]:
status = q("""
    SELECT order_status, COUNT(*) AS orders,
           100.0 * COUNT(*) / SUM(COUNT(*)) OVER () AS pct
    FROM raw.orders GROUP BY 1 ORDER BY 2 DESC
""")
display(status)
px.bar(status, x="order_status", y="orders", title="Orders by status").show()

## 7. Delivery date logic
**Look for:** delivered orders with no delivery date, delivery before purchase, and non-delivered orders that have a delivery date. These break on-time delivery calculations.

In [ ]:
delivery = q("""
    WITH o AS (
        SELECT order_id, order_status,
               TRY_CAST(order_purchase_timestamp AS TIMESTAMP) AS purchased,
               TRY_CAST(order_delivered_customer_date AS TIMESTAMP) AS delivered,
               TRY_CAST(order_estimated_delivery_date AS TIMESTAMP) AS estimated
        FROM raw.orders
    )
    SELECT
        COUNT(*) FILTER (WHERE order_status = 'delivered') AS delivered_orders,
        COUNT(*) FILTER (WHERE order_status = 'delivered' AND delivered IS NULL) AS delivered_missing_date,
        COUNT(*) FILTER (WHERE order_status <> 'delivered' AND delivered IS NOT NULL) AS not_delivered_but_has_date,
        COUNT(*) FILTER (WHERE delivered < purchased) AS delivered_before_purchase,
        COUNT(*) FILTER (WHERE order_status = 'delivered' AND delivered > estimated) AS late_orders,
        100.0 * COUNT(*) FILTER (WHERE order_status = 'delivered' AND delivered > estimated)
              / NULLIF(COUNT(*) FILTER (WHERE order_status = 'delivered' AND delivered IS NOT NULL), 0) AS late_pct
    FROM o
""")
display(delivery.T.rename(columns={0: "value"}))

days = q("""
    SELECT date_diff('day', TRY_CAST(order_purchase_timestamp AS TIMESTAMP),
                            TRY_CAST(order_delivered_customer_date AS TIMESTAMP)) AS delivery_days
    FROM raw.orders
    WHERE order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
""")
display(days["delivery_days"].describe())
px.histogram(days[days["delivery_days"].between(0, 60)], x="delivery_days", nbins=60,
             title="Delivery days (clipped to 0-60)").show()

## 8. Orders vs items vs payments
**Look for:** orders with no items, orders with no payment, orders paid in multiple installments/methods, and whether payment totals match item price + freight.
Payments are one-to-many per order, so they must be aggregated before joining to orders (otherwise revenue is double counted).

In [ ]:
coverage = q("""
    SELECT
        COUNT(*) AS orders,
        COUNT(*) FILTER (WHERE i.order_id IS NULL) AS orders_without_items,
        COUNT(*) FILTER (WHERE p.order_id IS NULL) AS orders_without_payments,
        COUNT(*) FILTER (WHERE r.order_id IS NULL) AS orders_without_reviews
    FROM raw.orders o
    LEFT JOIN (SELECT DISTINCT order_id FROM raw.order_items) i USING (order_id)
    LEFT JOIN (SELECT DISTINCT order_id FROM raw.order_payments) p USING (order_id)
    LEFT JOIN (SELECT DISTINCT order_id FROM raw.order_reviews) r USING (order_id)
""")
display(coverage)

display(q("""
    SELECT o.order_status, COUNT(*) AS orders_without_items
    FROM raw.orders o
    LEFT JOIN (SELECT DISTINCT order_id FROM raw.order_items) i USING (order_id)
    WHERE i.order_id IS NULL
    GROUP BY 1 ORDER BY 2 DESC
"""))

display(q("""
    SELECT payments_per_order, COUNT(*) AS orders FROM (
        SELECT order_id, COUNT(*) AS payments_per_order FROM raw.order_payments GROUP BY 1
    ) GROUP BY 1 ORDER BY 1
"""))

TOLERANCE = 1.0  # BRL; small gaps can come from vouchers and rounding
recon = q(f"""
    WITH items AS (
        SELECT order_id, SUM(TRY_CAST(price AS DOUBLE) + TRY_CAST(freight_value AS DOUBLE)) AS item_total
        FROM raw.order_items GROUP BY 1
    ), pay AS (
        SELECT order_id, SUM(TRY_CAST(payment_value AS DOUBLE)) AS pay_total
        FROM raw.order_payments GROUP BY 1
    )
    SELECT COUNT(*) AS orders_compared,
           COUNT(*) FILTER (WHERE ABS(item_total - pay_total) > {TOLERANCE}) AS mismatches_over_tolerance,
           100.0 * COUNT(*) FILTER (WHERE ABS(item_total - pay_total) > {TOLERANCE}) / COUNT(*) AS mismatch_pct
    FROM items JOIN pay USING (order_id)
""")
display(recon)

## 9. Reviews
**Look for:** score distribution, and orders with more than one review (we must pick one per order).

In [ ]:
scores = q("""
    SELECT review_score, COUNT(*) AS reviews FROM raw.order_reviews GROUP BY 1 ORDER BY 1
""")
display(scores)
px.bar(scores, x="review_score", y="reviews", title="Review score distribution").show()

display(q("""
    SELECT reviews_per_order, COUNT(*) AS orders FROM (
        SELECT order_id, COUNT(*) AS reviews_per_order FROM raw.order_reviews GROUP BY 1
    ) GROUP BY 1 ORDER BY 1
"""))

## 10. Products and categories
**Look for:** products with no category, categories missing an English translation, and zero/NULL weights.

In [ ]:
display(q("""
    SELECT
        COUNT(*) AS products,
        COUNT(*) FILTER (WHERE product_category_name IS NULL OR TRIM(product_category_name) = '') AS no_category,
        COUNT(DISTINCT product_category_name) AS distinct_categories,
        COUNT(*) FILTER (WHERE TRY_CAST(product_weight_g AS DOUBLE) IS NULL) AS weight_missing,
        COUNT(*) FILTER (WHERE TRY_CAST(product_weight_g AS DOUBLE) = 0) AS weight_zero
    FROM raw.products
"""))

display(q("""
    SELECT p.product_category_name AS untranslated_category, COUNT(*) AS products
    FROM raw.products p
    LEFT JOIN raw.category_translation t USING (product_category_name)
    WHERE t.product_category_name IS NULL AND p.product_category_name IS NOT NULL
      AND TRIM(p.product_category_name) <> ''
    GROUP BY 1 ORDER BY 2 DESC
"""))

display(q("""
    SELECT COALESCE(t.product_category_name_english, p.product_category_name, '(none)') AS category,
           COUNT(*) AS items
    FROM raw.order_items i
    JOIN raw.products p USING (product_id)
    LEFT JOIN raw.category_translation t USING (product_category_name)
    GROUP BY 1 ORDER BY 2 DESC LIMIT 15
"""))

## 11. Numeric outliers
**Look for:** zero or negative prices, extreme maximums, and a big gap between the 99th percentile and the maximum. We record these; we do not delete anything here.

In [ ]:
numeric_cols = [
    ("order_items", "price"),
    ("order_items", "freight_value"),
    ("order_payments", "payment_value"),
    ("products", "product_weight_g"),
]
out = []
for table, col in numeric_cols:
    out.append(
        q(f"""
            SELECT '{table}.{col}' AS field,
                   COUNT(*) AS n,
                   COUNT(*) FILTER (WHERE TRY_CAST({col} AS DOUBLE) IS NULL) AS unparseable,
                   COUNT(*) FILTER (WHERE TRY_CAST({col} AS DOUBLE) <= 0) AS zero_or_negative,
                   MIN(TRY_CAST({col} AS DOUBLE)) AS min,
                   quantile_cont(TRY_CAST({col} AS DOUBLE), 0.5) AS p50,
                   quantile_cont(TRY_CAST({col} AS DOUBLE), 0.99) AS p99,
                   MAX(TRY_CAST({col} AS DOUBLE)) AS max
            FROM raw.{table}
        """)
    )
display(pd.concat(out, ignore_index=True))

## 12. Geolocation
**Look for:** many rows per zip prefix. If so, this table cannot be joined directly (it would multiply rows) and must be reduced to one row per zip in staging.

In [ ]:
display(q("""
    SELECT COUNT(*) AS rows,
           COUNT(DISTINCT geolocation_zip_code_prefix) AS distinct_zip_prefixes,
           1.0 * COUNT(*) / COUNT(DISTINCT geolocation_zip_code_prefix) AS avg_rows_per_zip
    FROM raw.geolocation
"""))

## 13. Wrap-up

Copy every surprise into `docs/data_findings.md` using this format:

| # | Issue | Evidence (numbers from this notebook) | Decision | Implemented in |
|---|---|---|---|---|

Then tell Claude the numbers from sections 3, 4, 5, 7 and 8; they drive the Phase 4 staging models.